In [20]:
from google.cloud import bigquery
import glob
import json

import re

### Transformation (va dépendre de mon schéma en étoile)

In [37]:
client = bigquery.Client.from_service_account_json("../jan26-cde-crypto-cab74f95479e.json")

table_id = "jan26-cde-crypto.db_crypto.STAGING"

In [46]:
data = []
for path in glob.glob("../etape_1_exploration_api/*.json"):
    reg = re.compile(r'(BTCUSDT|ETHUSDT|BNBUSDT)_(\d\w)')
    crypto, interval = reg.findall(path)[0]
    with open(path, "r") as f:
        data_temp = json.load(f)

        for doc in data_temp:
            doc["interval"] = interval
            doc["crypto"] = crypto
        data.extend(data_temp)

rows_to_insert = [{"raw_data": doc} for doc in data]
   

In [47]:
rows_to_insert[0:2]

[{'raw_data': {'open_time': 1502928000000,
   'open': 4261.48,
   'high': 4485.39,
   'low': 4200.74,
   'close': 4285.08,
   'volume': 795.150377,
   'interval': '1d',
   'crypto': 'BTCUSDT'}},
 {'raw_data': {'open_time': 1503014400000,
   'open': 4285.08,
   'high': 4371.52,
   'low': 3938.77,
   'close': 4108.37,
   'volume': 1199.888264,
   'interval': '1d',
   'crypto': 'BTCUSDT'}}]

In [54]:
job = client.load_table_from_json(
    rows_to_insert,
    table_id,
)

job.result()
print("OK")


OK
